# LangGraph Travel Planner with Checkpoints

## 1. Project Overview

**Task:** Build a **LangGraph workflow** that gathers travel preferences, generates itinerary options, pauses for approval, and refines the selected plan.

This notebook is designed as a **beginner-friendly LangGraph project**. The focus is not only on generating itineraries, but on understanding how **state graphs**, **approval checkpoints**, and **revision loops** work together in a practical planning assistant.

### Workflow

```
  START
    │
    ▼
  gather_preferences
    │   collect destination, budget, duration, pace, interests
    ▼
  generate_options
    │   create 2-3 itinerary options
    ▼
  approval_checkpoint
   ├── approve  ───────────────► finalize_plan
   ├── refine   ───────────────► refine_plan
   └── reject   ───────────────► generate_options
                                  ▲
                                  │
                              approval_checkpoint
```

### What You Will Learn

- How to model a multi-step planning workflow as a **state graph**
- How to use **TypedDict state** to pass structured data across nodes
- How to create **approval gates** for human-in-the-loop workflows
- How to implement **refinement loops** without losing earlier context
- How to store **revision checkpoints** so the user can review earlier versions

### Why This Project Matters

Many real-world GenAI products do not just generate one final answer. They:

1. collect requirements
2. generate candidate outputs
3. pause for review or approval
4. refine based on feedback
5. save checkpoints for traceability

Travel planning is a simple domain for demonstrating this pattern, but the same design also applies to **content review, procurement, support routing, and planning agents**.

## 2. LangGraph Concepts Used Here

Before writing code, separate these four ideas:

### A. State
The state is the shared data structure that moves through the graph.

In this notebook, the state stores:
- traveler preferences
- generated itinerary options
- approval decision
- feedback for refinement
- the final plan
- revision checkpoints

### B. Nodes
Each node performs one job:
- gather preferences
- generate options
- request approval
- refine a plan
- finalize output

### C. Edges
Edges define the order of execution. A normal edge always moves to the next node. A conditional edge chooses the next node based on the current state.

### D. Checkpoints
A checkpoint is a stored snapshot of planning state at a particular stage.

In this notebook, checkpoints are educational and explicit:
- after generating itinerary options
- after each approval or refinement round
- before finalization

This makes it easy to answer questions like:
- What did option set 1 look like?
- What feedback caused the second revision?
- Which version was finally approved?


## 3. Environment Setup

In [ ]:
# Uncomment if you need to install packages
# !pip install -q langchain langchain-ollama langchain-core langgraph pandas

print("Dependencies: langchain, langgraph, pandas")
print("Model: Ollama with qwen3.5:9b")

## 4. Imports & Configuration

In [ ]:
import os
import re
import json
import textwrap
import warnings
from typing import TypedDict
from datetime import datetime

import pandas as pd

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

LLM_MODEL = "qwen3.5:9b"
print(f"Using model: {LLM_MODEL}")

## 5. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.3) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 100):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


def make_checkpoint(stage: str, payload: dict) -> dict:
    return {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "stage": stage,
        "payload": payload,
    }


test = ask("Say 'ready'.")
print(f"LLM check: {test[:30]}")

---

# Part A — Example Travel Requests & State Schema

## 6. Example Traveler Profiles

These examples let us run the workflow end-to-end without interactive form entry.

In [ ]:
TRAVEL_REQUESTS = [
    {
        "trip_id": "TRIP-001",
        "traveler_name": "Aisha",
        "destination": "Tokyo",
        "duration_days": 5,
        "budget_level": "medium",
        "travel_style": "culture + food + walkable neighborhoods",
        "pace": "balanced",
        "must_haves": ["one day trip option", "anime shopping area", "great ramen"],
        "constraints": ["no overly early starts", "hotel near public transit"],
        "approval_simulation": "refine",
        "feedback": "Make it slightly less packed and include one quieter evening option.",
    },
    {
        "trip_id": "TRIP-002",
        "traveler_name": "Daniel",
        "destination": "Barcelona",
        "duration_days": 4,
        "budget_level": "high",
        "travel_style": "architecture + local food + relaxed pace",
        "pace": "relaxed",
        "must_haves": ["Gaudi highlights", "seafood dinner", "beach time"],
        "constraints": ["afternoon breaks", "premium hotel option"],
        "approval_simulation": "approve",
        "feedback": "Looks good. Keep it as is.",
    },
    {
        "trip_id": "TRIP-003",
        "traveler_name": "Mina",
        "destination": "Istanbul",
        "duration_days": 6,
        "budget_level": "budget",
        "travel_style": "history + markets + scenic ferry views",
        "pace": "active",
        "must_haves": ["historic sites", "street food", "Bosporus ferry"],
        "constraints": ["keep transport simple", "hostel or affordable hotel"],
        "approval_simulation": "reject",
        "feedback": "The first set feels too generic. Give me more market-focused options and stronger budget tradeoffs.",
    },
]

print(f"Travel requests: {len(TRAVEL_REQUESTS)}")
for req in TRAVEL_REQUESTS:
    print(f"  {req['trip_id']} | {req['traveler_name']} | {req['destination']} | {req['duration_days']} days")

## 7. State Schema

This workflow stores both the **latest planning artifacts** and the **checkpoint history**.

### State Fields

- `trip_request`: raw structured input
- `preferences_summary`: normalized planning brief
- `itinerary_options`: candidate itineraries
- `selected_option`: one chosen option after approval
- `approval_decision`: approve / refine / reject
- `approval_feedback`: human feedback or simulation feedback
- `refined_plan`: revised itinerary
- `revision_count`: loop counter
- `checkpoints`: stored snapshots of important planning stages
- `final_plan`: final formatted itinerary

In [ ]:
class TravelState(TypedDict):
    trip_request: dict
    preferences_summary: str
    itinerary_options: str
    selected_option: str
    approval_decision: str
    approval_feedback: str
    refined_plan: str
    revision_count: int
    checkpoints: list
    final_plan: str
    current_node: str


print("State schema: TravelState")
for field, typ in TravelState.__annotations__.items():
    print(f"  {field}: {typ}")

---

# Part B — Build the Workflow Nodes

## 8. Node 1: Gather Preferences

This node converts raw trip input into a compact planning brief.

### Why not pass raw input everywhere?
A short normalized summary makes later prompts more consistent and easier to debug.

### Transition
- **Input:** `trip_request`
- **Output:** `preferences_summary`

In [ ]:
PREFERENCES_SYSTEM = """You are a travel planning assistant. Convert the raw traveler request
into a concise planning brief.

Include:
- destination
- trip length
- budget
- travel style
- pace
- must-haves
- constraints
- planning priorities

Write clearly and compactly for downstream itinerary generation."""


def gather_preferences(state: TravelState) -> dict:
    print("  [NODE] gather_preferences")
    req = state["trip_request"]

    summary = ask(
        f"Traveler request:\n{json.dumps(req, indent=2)}\n\nCreate a planning brief.",
        system=PREFERENCES_SYSTEM,
        temperature=0.2,
    )

    print(f"    Preferences summary: {len(summary)} chars")
    return {
        "preferences_summary": summary,
        "current_node": "gather_preferences",
    }


print("Node defined: gather_preferences")
print("  READS:  trip_request")
print("  WRITES: preferences_summary")

## 9. Node 2: Generate Itinerary Options

This node creates multiple candidate plans instead of forcing one answer immediately.

### Why multiple options?
A planning assistant should present tradeoffs:
- one option may maximize landmarks
- another may optimize budget
- another may favor a relaxed pace

### Checkpoint
After this node, we store a checkpoint of the generated options so we can compare revisions later.

In [ ]:
OPTIONS_SYSTEM = """You are a travel itinerary designer. Create 3 distinct itinerary options
for the traveler.

For each option include:
- option name
- theme / planning style
- day-by-day outline
- estimated budget level
- best fit explanation

Make the options meaningfully different. Use clear headings."""


def generate_options(state: TravelState) -> dict:
    print("  [NODE] generate_options")
    req = state["trip_request"]

    options = ask(
        f"TRAVEL BRIEF:\n{state['preferences_summary']}\n\n"
        f"Traveler: {req['traveler_name']}\n"
        f"Generate 3 itinerary options.",
        system=OPTIONS_SYSTEM,
        temperature=0.5,
    )

    checkpoints = list(state.get("checkpoints", []))
    checkpoints.append(
        make_checkpoint(
            "generate_options",
            {
                "revision_count": state.get("revision_count", 0),
                "itinerary_options": options,
            },
        )
    )

    print(f"    Options: {len(options)} chars")
    print(f"    Checkpoints: {len(checkpoints)}")

    return {
        "itinerary_options": options,
        "checkpoints": checkpoints,
        "current_node": "generate_options",
    }


print("Node defined: generate_options")
print("  READS:  preferences_summary, trip_request")
print("  WRITES: itinerary_options, checkpoints")

## 10. Node 3: Approval Checkpoint

This is the human-in-the-loop step. In a production app, this would pause and wait for user input.

In this notebook, we **simulate approval** using fields stored in the example request.

### Possible outcomes
- `approve`: accept one option and finalize
- `refine`: revise the plan with feedback
- `reject`: discard options and generate a fresh set

In [ ]:
def approval_checkpoint(state: TravelState) -> dict:
    print("  [NODE] approval_checkpoint")
    req = state["trip_request"]

    decision = req.get("approval_simulation", "approve")
    feedback = req.get("feedback", "")

    selected_option = "Option 1"
    if decision == "refine":
        selected_option = "Option 2"
    elif decision == "reject":
        selected_option = "None yet"

    checkpoints = list(state.get("checkpoints", []))
    checkpoints.append(
        make_checkpoint(
            "approval_checkpoint",
            {
                "decision": decision,
                "selected_option": selected_option,
                "feedback": feedback,
                "revision_count": state.get("revision_count", 0),
            },
        )
    )

    print(f"    Decision: {decision.upper()}")
    print(f"    Selected option: {selected_option}")

    return {
        "approval_decision": decision,
        "approval_feedback": feedback,
        "selected_option": selected_option,
        "checkpoints": checkpoints,
        "current_node": "approval_checkpoint",
    }


print("Node defined: approval_checkpoint")
print("  READS:  trip_request, itinerary_options")
print("  WRITES: approval_decision, approval_feedback, selected_option, checkpoints")

## 11. Node 4: Refine Plan

This node is called only when the reviewer asks for adjustments. It preserves the earlier options and creates a refined plan using feedback.

### Why keep the original options?
Because refinement is not the same as regeneration. We want a traceable sequence:
1. original options
2. approval feedback
3. revised version

In [ ]:
REFINE_SYSTEM = """You are refining a travel itinerary after reviewer feedback.

Take the existing options and feedback, then produce a revised itinerary.

Requirements:
- keep the good parts of the selected option
- explicitly apply the feedback
- produce a polished day-by-day itinerary
- include why the revision is better

Use clear headings and practical travel structure."""


def refine_plan(state: TravelState) -> dict:
    print("  [NODE] refine_plan")
    revision_count = state.get("revision_count", 0) + 1

    refined = ask(
        f"PREFERENCES:\n{state['preferences_summary']}\n\n"
        f"ORIGINAL OPTIONS:\n{state['itinerary_options'][:3000]}\n\n"
        f"SELECTED OPTION: {state['selected_option']}\n"
        f"FEEDBACK: {state['approval_feedback']}\n\n"
        f"Refine the travel plan.",
        system=REFINE_SYSTEM,
        temperature=0.4,
    )

    checkpoints = list(state.get("checkpoints", []))
    checkpoints.append(
        make_checkpoint(
            "refine_plan",
            {
                "revision_count": revision_count,
                "feedback": state['approval_feedback'],
                "refined_plan": refined,
            },
        )
    )

    print(f"    Revision count: {revision_count}")
    print(f"    Refined plan: {len(refined)} chars")

    return {
        "refined_plan": refined,
        "revision_count": revision_count,
        "checkpoints": checkpoints,
        "current_node": "refine_plan",
    }


print("Node defined: refine_plan")
print("  READS:  preferences_summary, itinerary_options, selected_option, approval_feedback")
print("  WRITES: refined_plan, revision_count, checkpoints")

## 12. Node 5: Finalize Plan

This node creates the final user-facing itinerary.

If a refined plan exists, it becomes the basis for the final output. Otherwise, the approved option set is finalized directly.

In [ ]:
FINALIZE_SYSTEM = """You are formatting the final approved travel plan.

Produce a polished final itinerary with:
- trip overview
- day-by-day plan
- budget/style fit note
- practical planning reminders
- summary of any refinement changes

Keep the output clean, readable, and ready to share."""


def finalize_plan(state: TravelState) -> dict:
    print("  [NODE] finalize_plan")
    source_plan = state.get("refined_plan") or state.get("itinerary_options")

    final = ask(
        f"TRAVEL BRIEF:\n{state['preferences_summary']}\n\n"
        f"SELECTED OPTION: {state.get('selected_option', 'Option 1')}\n\n"
        f"PLAN TO FINALIZE:\n{source_plan[:3500]}\n\n"
        f"REVISION COUNT: {state.get('revision_count', 0)}\n"
        f"APPROVAL FEEDBACK: {state.get('approval_feedback', 'None')}\n\n"
        f"Create the final itinerary.",
        system=FINALIZE_SYSTEM,
        temperature=0.2,
    )

    checkpoints = list(state.get("checkpoints", []))
    checkpoints.append(
        make_checkpoint(
            "finalize_plan",
            {
                "selected_option": state.get("selected_option", "Option 1"),
                "revision_count": state.get("revision_count", 0),
                "final_plan": final,
            },
        )
    )

    print(f"    Final plan: {len(final)} chars")

    return {
        "final_plan": final,
        "checkpoints": checkpoints,
        "current_node": "finalize_plan",
    }


print("Node defined: finalize_plan")
print("  READS:  preferences_summary, itinerary_options/refined_plan, selected_option")
print("  WRITES: final_plan, checkpoints")

---

# Part C — Conditional Routing

## 13. Approval Routing Logic

LangGraph uses a routing function to decide which node runs next after approval.

### Routing Rules
- `approve` → `finalize_plan`
- `refine` → `refine_plan`
- `reject` → `generate_options`

### Why reject goes back to generate_options
Refinement means: *improve the current direction*.
Rejection means: *start over with a different set of options*.

In [ ]:
MAX_REVISIONS = 2


def route_after_approval(state: TravelState) -> str:
    decision = state.get("approval_decision", "approve")
    revision_count = state.get("revision_count", 0)

    if decision == "approve":
        print("    [ROUTE] approve -> finalize_plan")
        return "finalize_plan"

    if decision == "refine":
        if revision_count >= MAX_REVISIONS:
            print(f"    [ROUTE] refine requested but cap reached ({MAX_REVISIONS}) -> finalize_plan")
            return "finalize_plan"
        print("    [ROUTE] refine -> refine_plan")
        return "refine_plan"

    print("    [ROUTE] reject -> generate_options")
    return "generate_options"


def route_after_refinement(state: TravelState) -> str:
    print("    [ROUTE] after refinement -> finalize_plan")
    return "finalize_plan"


print(f"Routing helpers ready (max revisions: {MAX_REVISIONS})")

## 14. Build the LangGraph Workflow

In [ ]:
workflow = StateGraph(TravelState)

workflow.add_node("gather_preferences", gather_preferences)
workflow.add_node("generate_options", generate_options)
workflow.add_node("approval_checkpoint", approval_checkpoint)
workflow.add_node("refine_plan", refine_plan)
workflow.add_node("finalize_plan", finalize_plan)

workflow.add_edge(START, "gather_preferences")
workflow.add_edge("gather_preferences", "generate_options")
workflow.add_edge("generate_options", "approval_checkpoint")

workflow.add_conditional_edges(
    "approval_checkpoint",
    route_after_approval,
    {
        "finalize_plan": "finalize_plan",
        "refine_plan": "refine_plan",
        "generate_options": "generate_options",
    },
)

workflow.add_conditional_edges(
    "refine_plan",
    route_after_refinement,
    {
        "finalize_plan": "finalize_plan",
    },
)

workflow.add_edge("finalize_plan", END)

app = workflow.compile()

print("Graph compiled.")
print()
print("Nodes:")
for node_name in workflow.nodes.keys():
    print(f"  - {node_name}")

In [ ]:
try:
    mermaid_str = app.get_graph().draw_mermaid()
    print("Mermaid diagram:")
    print(mermaid_str)
except Exception as e:
    print(f"Mermaid rendering unavailable: {e}")
    print("\nText graph:")
    print("  START -> gather_preferences -> generate_options -> approval_checkpoint")
    print("  approval_checkpoint -> finalize_plan")
    print("  approval_checkpoint -> refine_plan -> finalize_plan")
    print("  approval_checkpoint -> generate_options (reject loop)")

---

# Part D — Run the Workflow

## 15. Initialize State

In [ ]:
def make_initial_state(request: dict) -> TravelState:
    return {
        "trip_request": request,
        "preferences_summary": "",
        "itinerary_options": "",
        "selected_option": "",
        "approval_decision": "",
        "approval_feedback": "",
        "refined_plan": "",
        "revision_count": 0,
        "checkpoints": [],
        "final_plan": "",
        "current_node": "start",
    }


sample_request = TRAVEL_REQUESTS[0]
print(f"Sample trip: {sample_request['trip_id']} -> {sample_request['destination']}")
print(f"Approval simulation: {sample_request['approval_simulation']}")

## 16. Run Example 1 — Refinement Path

In [ ]:
result_1 = app.invoke(make_initial_state(sample_request), {"recursion_limit": 20})

print("Workflow complete.")
print(f"  Current node: {result_1['current_node']}")
print(f"  Approval decision: {result_1['approval_decision']}")
print(f"  Revision count: {result_1['revision_count']}")
print(f"  Checkpoints: {len(result_1['checkpoints'])}")
print(f"  Final plan length: {len(result_1['final_plan'])} chars")

## 17. View the Final Plan

In [ ]:
print(f"FINAL PLAN — {sample_request['destination']}")
print("=" * 70)
wrap_print(result_1['final_plan'])

## 18. Inspect Checkpoints

Checkpoints are simple stored snapshots. They help us understand the planning history.

In [ ]:
print("CHECKPOINT HISTORY")
print("=" * 70)

for i, checkpoint in enumerate(result_1['checkpoints'], 1):
    print(f"\nCheckpoint {i}: {checkpoint['stage']}")
    print(f"  Time: {checkpoint['timestamp']}")
    payload = checkpoint['payload']
    for key, value in payload.items():
        if isinstance(value, str):
            preview = value[:120].replace('\n', ' ')
            print(f"  {key}: {preview}{'...' if len(value) > 120 else ''}")
        else:
            print(f"  {key}: {value}")

## 19. Streaming Execution

Streaming makes the workflow easier to understand because you can watch state updates step by step.

In [ ]:
print("STREAMING EXECUTION — Example 1")
print("=" * 70)

step = 0
for chunk in app.stream(make_initial_state(sample_request), {"recursion_limit": 20}):
    step += 1
    for node_name, node_output in chunk.items():
        details = []
        for key, value in node_output.items():
            if key == "current_node":
                continue
            if isinstance(value, str):
                details.append(f"{key}: {len(value)} chars")
            elif isinstance(value, list):
                details.append(f"{key}: {len(value)} items")
            else:
                details.append(f"{key}: {value}")
        print(f"  Step {step}: {node_name:<22} -> {', '.join(details)}")

---

# Part E — Compare Multiple Approval Outcomes

## 20. Run All Three Scenarios

In [ ]:
all_results = []

for req in TRAVEL_REQUESTS:
    print(f"\n{'=' * 70}")
    print(f"Running {req['trip_id']} | {req['destination']} | approval={req['approval_simulation']}")
    print(f"{'=' * 70}")
    result = app.invoke(make_initial_state(req), {"recursion_limit": 25})
    all_results.append(result)
    print(f"  Finalized with {len(result['checkpoints'])} checkpoints and {result['revision_count']} revisions")

## 21. Compare Workflow Outcomes

This table helps explain how conditional routing changes the execution path.

In [ ]:
df = pd.DataFrame([
    {
        "Trip ID": req['trip_id'],
        "Destination": req['destination'],
        "Approval Sim": req['approval_simulation'],
        "Revision Count": result['revision_count'],
        "Checkpoints": len(result['checkpoints']),
        "Final Plan Size": len(result['final_plan']),
    }
    for req, result in zip(TRAVEL_REQUESTS, all_results)
])

print(df.to_string(index=False))

## 22. Approve vs Refine vs Reject Explained

### Approve
The plan is good enough, so the graph moves directly to finalization.

### Refine
The plan has a strong direction, but needs edits. The graph uses feedback to revise it.

### Reject
The plan is off-target, so the graph loops back to generate a fresh set of options.

This difference is important because refinement preserves earlier structure, while rejection asks the system to rethink the option set.

In [ ]:
for req, result in zip(TRAVEL_REQUESTS, all_results):
    print(f"\n{'#' * 80}")
    print(f"# {req['trip_id']} | {req['destination']} | decision={result['approval_decision']}")
    print(f"{'#' * 80}")
    wrap_print(result['final_plan'][:1200])
    if len(result['final_plan']) > 1200:
        print(f"  ... [{len(result['final_plan']) - 1200} more chars]")

---

# Part F — Checkpoints & Revision Analysis

## 23. Why Checkpoints Matter

Checkpoints are not the same as full LangGraph persistence. Here they are **application-level checkpoints** stored in state.

That means they are useful for:
- revision history
- auditability
- debugging
- showing the user what changed

But they do **not** automatically survive process restarts.

If you wanted true cross-session persistence, you would add a LangGraph checkpointer such as `MemorySaver`, `SqliteSaver`, or `PostgresSaver`.

In [ ]:
print("CHECKPOINT COUNTS BY TRIP")
print("=" * 70)

for req, result in zip(TRAVEL_REQUESTS, all_results):
    print(f"  {req['trip_id']} | {req['destination']:<10} | checkpoints={len(result['checkpoints'])} | revisions={result['revision_count']}")

print()
print("Interpretation:")
print("  - More checkpoints usually mean more review or revision stages.")
print("  - A direct approval path stores fewer checkpoints than a refine/reject path.")

## 24. Visualize a Single Revision Trace

In [ ]:
trace = all_results[0]['checkpoints']

print("REVISION TRACE — Example 1")
print("=" * 70)

for i, cp in enumerate(trace, 1):
    print(f"{i}. {cp['stage']} @ {cp['timestamp']}")
    payload = cp['payload']
    if 'revision_count' in payload:
        print(f"   revision_count: {payload['revision_count']}")
    if 'decision' in payload:
        print(f"   decision: {payload['decision']}")
    if 'feedback' in payload and payload['feedback']:
        print(f"   feedback: {payload['feedback'][:100]}")

---

# Part G — Design Notes & Exercises

## 25. Beginner Design Notes

### Why use LangGraph here instead of one prompt?
Because the task has **stages and decisions**:
- collect preferences
- generate multiple options
- pause for approval
- refine if needed
- finalize

This is more naturally represented as a workflow than as a single prompt.

### Why put approval in the graph?
Approval changes the next action. That is exactly what conditional routing is for.

### Why store checkpoints in state?
Because travel planning is iterative. Checkpoints make revisions visible instead of hidden.

## 26. Exercises

### Exercise 1: Add hotel suggestions
Create a new node called `recommend_stays` after `generate_options`. It should suggest 2-3 lodging styles based on budget and trip style.

### Exercise 2: Add a transportation node
Insert a node that recommends airport transfer and local transportation choices before finalization.

### Exercise 3: Make approval interactive
Replace `approval_simulation` with a real `input()` prompt so the notebook pauses and asks the user whether to approve, refine, or reject.

### Exercise 4: Persist checkpoints to disk
Write the `checkpoints` list to a JSON file after each run, then reload it in a later session to compare versions.

### Exercise 5: Compare one-shot planning vs workflow planning
Create a simple baseline prompt that generates one itinerary directly. Then compare it with the LangGraph workflow in terms of clarity, flexibility, and revisability.

In [ ]:
print("SESSION SUMMARY")
print("=" * 70)
print(f"Trips processed: {len(all_results)}")
print(f"Average checkpoints: {df['Checkpoints'].mean():.1f}")
print(f"Average revisions: {df['Revision Count'].mean():.1f}")
print()
print("Workflow pattern:")
print("  gather_preferences -> generate_options -> approval_checkpoint")
print("  approval -> finalize")
print("  refine -> refine_plan -> finalize")
print("  reject -> generate_options (loop)")
print()
print("What checkpoints captured:")
print("  - generated option sets")
print("  - approval decisions and feedback")
print("  - refined itinerary versions")
print("  - final approved plan")

## 27. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **LangGraph is useful when a task has stages and decisions** |
| 2 | **Approval gates are natural conditional routing points** |
| 3 | **Refinement is different from rejection** — refine improves, reject restarts |
| 4 | **Checkpoints make planning traceable** |
| 5 | **State carries context across nodes** so prompts stay coordinated |
| 6 | **Multiple options are better than one-shot plans** for user review workflows |
| 7 | **Revision limits matter** so loops stay controlled |
| 8 | **Application-level checkpoints and LangGraph persistence are different ideas** |
| 9 | **This pattern generalizes** to approval-heavy AI apps beyond travel |
| 10 | **Notebook workflows make GenAI systems easier to learn and debug** |

---

*This notebook is part of the Machine Learning Projects repository.*